In [33]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import  MinMaxScaler


In [34]:
# load dataset
df = pd.read_csv('raw_loan_dataset.csv')
print(df.head())

    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  


In [35]:
# dataset info
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB
None


In [36]:
# missing value
print(df.isnull().sum())

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


In [37]:
#clean currancy formating 
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)
print("currency cleaning completed")
    

currency cleaning completed


In [38]:
# fix category errors before impute
map = {   "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",}


In [39]:
df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(map).replace({"nan": np.nan})

In [40]:
# Impute missing values

df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

In [41]:
# 5) Remove duplicates

before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")



Dropped duplicates: 103 -> 100 rows


In [42]:
# outlier
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)
print("outlier successfully caped")

outlier successfully caped


In [43]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))


Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [44]:
# class balance
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)


Class balance OK for baseline Accuracy (both classes well represented).


In [45]:
# feature enginerring
df['DebtToIncome']= df['LoanAmount'] / df['Income']
df['IcomePerYearEmployed'] = df['Income'] / (df['EmploymentYears'] + 1) 

In [46]:
# feature  scaling
Scaler = MinMaxScaler()
numeric_feature = ['Income' , 'CreditScore', 'LoanAmount' ,'EmploymentYears',  'DebtToIncome', 'IcomePerYearEmployed' ]
numeric_feature = Scaler.fit_transform(df[numeric_feature])
print("scaling completed successfully using minmaxscaler")

scaling completed successfully using minmaxscaler


In [49]:
# final check
df.to_csv('Clean_loan_dataset.csv', index = False)
print("dataset saved to 'Clean_loan_dataset.csv'.pipline completed")

dataset saved to 'Clean_loan_dataset.csv'.pipline completed


In [50]:
print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0              1   
1   96482.0        524.0             15.0     11200.0              1   
2   28478.0        638.0              5.4     12100.0              0   
3   25851.0        561.0             17.6      7000.0              1   
4   38396.0        527.0              9.8     10700.0              0   

   PreviousDefaults  Approved  DebtToIncome  IcomePerYearEmployed  
0                 0         0      0.237111          51814.285714  
1                 0         1      0.116084           6030.125000  
2                 0         1      0.424889           4449.687500  
3                 0         1      0.270783           1389.838710  
4                 0         1      0.278675           3555.185185  
